In [2]:
import muspan as ms
import numpy as np  
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

# path_to_pts = '../data/data_with_centroids/Muspan_data/AKPT_Liver_Mets/'
path_to_pts = '../domains_with_mets/'

centers = {
    "Sample_4d_1": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_4d_1_Region_1_converted.muspan'),
    "Sample_4d_1_2": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_4d_1_Region_2_converted.muspan'),
    "Sample_4d_2": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_4d_2_Region_1_converted.muspan'),
    "Sample_4d_3": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_4d_3_Region_1_converted.muspan'),
    # "Sample_4d_4": ms.io.load_domain(path_to_pts + 'Sample_4d_4_converted.muspan'),
    "Sample_4d_5": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_4d_5_Region_1_converted.muspan'),
    "Sample_28d_1": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_28d_1_Region_1_converted.muspan'),
    "Sample_28d_1_2": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_28d_1_Region_2_converted.muspan'),
    "Sample_28d_2": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_28d_2_Region_1_converted.muspan'),
    "Sample_28d_2_2": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_28d_2_Region_2_converted.muspan'),
    "Sample_28d_3": ms.io.load_domain(path_to_pts + 'AKPT_Liver_Mets_28d_3_Region_1_converted.muspan')
    # # "Sample_Primary_1": ms.io.load_domain(path_to_pts + 'Sample_Primary_1_converted.muspan'),
    # # "Sample_Primary_1_2": ms.io.load_domain(path_to_pts + 'Sample_Primary_1_2_converted.muspan'),
    # # "Sample_Primary_2": ms.io.load_domain(path_to_pts + 'Sample_Primary_2_converted.muspan'),
    # # "Sample_Primary_2_2": ms.io.load_domain(path_to_pts + 'Sample_Primary_2_2_converted.muspan'),
}

output_path = '../outputs/In_out'

# accumulate per-met counts
min_cells = 15

# helper: cells contained in a given met (uses your original query style)
def cell_in_met(domain, cells_query, met_id):
    q1 = ms.query.query(domain, ('contains', (cells_query, [met_id])), 'is', True) 
    # q2 = ms.query.query(domain, ('distance', ('boundary', cells_query, [met_id])), 'is', 0)
    # return ms.query.query_container(q1, 'OR', q2, domain)
    return q1

for sample_name, domain in centers.items():
    print(f"\n=== Processing {sample_name} ===")

    # run a query for just the mets
    mets = ms.query.query(domain, ('collection',), 'is', 'Detailed Met Annotations')
    ids = ms.query.return_object_IDs_from_query_like(domain, mets)
    query_centers = ms.query.query(domain, ('collection',), 'is', 'Cell centres')
    query_boundaries = ms.query.query(domain, ('collection',), 'is', 'Cell boundaries')

    # query that is only the cells within the domain
    cells = ms.query.query(domain, ('collection',), 'is', 'Cell centres')

    # remove any mets with total cell numbers < 1 (keep >=1)
    keep_mets = []
    cells_in_met_ids = []
    met_to_cell_centroid_ids = {}
    # for met_id in ids:
    for met_id_number in range(len(ids)):
        met_id = ids[met_id_number]
        cells_in_met = []
        query_cells_in_met = cell_in_met(domain, cells, met_id)
        ids_in = ms.query.return_object_IDs_from_query_like(domain, query_cells_in_met)
        if len(ids_in) >= min_cells:
            keep_mets.append(met_id)
            met_to_cell_centroid_ids[met_id] = ids_in
        for i in ids_in:
            index = np.where(domain.labels['Cell ID']['objects'] == i)[0][0]
            cells_in_met.append(domain.labels['Cell ID']['labels'][index])

        test = ms.query.query(domain, ('label', 'Cell ID'), 'in', cells_in_met)

        query_container_boundary = ms.query.query_container(test, 'AND', query_boundaries, domain)
        query_container_center = ms.query.query_container(test, 'AND', query_centers, domain)

        domain.add_objects_to_collection(query_container_boundary, 'Cell Boundaries in Met ' + str(met_id_number+1))
        domain.add_objects_to_collection(query_container_center, 'Cell Centres in Met ' + str(met_id_number+1))
        domain.add_objects_to_collection([met_id], 'Metastasis ' + str(met_id_number+1))

        cells_in_met_ids.append(cells_in_met) # in same order as ids
        print(ms.query.return_object_IDs_from_query_like(domain, test))
    print(f"Keeping {len(keep_mets)} mets with at least {min_cells} cells")
    ms.io.save_domain(domain, name_of_file = sample_name+'.muspan', path_to_save = output_path)

MuSpAn domain loaded successfully. Domain summary:
Domain name: AKPT_Liver_Mets_4d_1_Region_1
Number of objects: 49597
Collections: ['Cell boundaries', 'Met annotations', 'Cell centres', 'Detailed Met Annotations']
Labels: ['Cell ID', 'Transcript Counts', 'Cell Area', 'Cluster ID', 'Nucleus Area', 'Transcript Counts: 1700019D03Rik', 'Transcript Counts: Abcc8', 'Transcript Counts: Acta2', 'Transcript Counts: Acvr1b', 'Transcript Counts: Acvr2a', 'Transcript Counts: Adgra2', 'Transcript Counts: Adgre1', 'Transcript Counts: Adh1', 'Transcript Counts: Adra2a', 'Transcript Counts: Afap1l2', 'Transcript Counts: Alb', 'Transcript Counts: Alcam', 'Transcript Counts: Aldh1b1', 'Transcript Counts: Aldoa', 'Transcript Counts: Amotl2', 'Transcript Counts: Ano7', 'Transcript Counts: Anxa1', 'Transcript Counts: Anxa10', 'Transcript Counts: Aqp1', 'Transcript Counts: Aqp8', 'Transcript Counts: Areg', 'Transcript Counts: Arhgap24', 'Transcript Counts: Ascl2', 'Transcript Counts: Asl', 'Transcript Coun

In [2]:
checking = {
    "Sample_28d_2_2_check": ms.io.load_domain(output_path + 'Sample_28d_2_2.muspan')}

domain = checking["Sample_28d_2_2_check"]
print(domain.collections)

MuSpAn domain loaded successfully. Domain summary:
Domain name: AKPT_Liver_Mets_28d_2_Region_2
Number of objects: 324576
Collections: ['Cell boundaries', 'Cell centres', 'Detailed Met Annotations', 'Cell Boundaries in Met 1', 'Cell Centres in Met 1', 'Metastasis 1', 'Cell Boundaries in Met 2', 'Cell Centres in Met 2', 'Metastasis 2', 'Cell Boundaries in Met 3', 'Cell Centres in Met 3', 'Metastasis 3', 'Cell Boundaries in Met 4', 'Cell Centres in Met 4', 'Metastasis 4', 'Cell Boundaries in Met 5', 'Cell Centres in Met 5', 'Metastasis 5', 'Cell Boundaries in Met 6', 'Cell Centres in Met 6', 'Metastasis 6', 'Cell Boundaries in Met 7', 'Cell Centres in Met 7', 'Metastasis 7']
Labels: ['Cell ID', 'Transcript Counts', 'Cell Area', 'Cluster ID', 'Nucleus Area', 'Transcript Counts: 1700019D03Rik', 'Transcript Counts: Abcc8', 'Transcript Counts: Acta2', 'Transcript Counts: Acvr1b', 'Transcript Counts: Acvr2a', 'Transcript Counts: Adgra2', 'Transcript Counts: Adgre1', 'Transcript Counts: Adh1', 

In [3]:
domain.collections.keys()

dict_keys(['Cell boundaries', '__temporary_shape_cache', 'Cell centres', 'Detailed Met Annotations', 'Cell Boundaries in Met 1', 'Cell Centres in Met 1', 'Metastasis 1', 'Cell Boundaries in Met 2', 'Cell Centres in Met 2', 'Metastasis 2', 'Cell Boundaries in Met 3', 'Cell Centres in Met 3', 'Metastasis 3', 'Cell Boundaries in Met 4', 'Cell Centres in Met 4', 'Metastasis 4', 'Cell Boundaries in Met 5', 'Cell Centres in Met 5', 'Metastasis 5', 'Cell Boundaries in Met 6', 'Cell Centres in Met 6', 'Metastasis 6', 'Cell Boundaries in Met 7', 'Cell Centres in Met 7', 'Metastasis 7'])